In [1]:
import cv2, time, torch
from PIL import Image
from transformers import AutoProcessor, AutoModel, AutoTokenizer
from pathlib import Path

c:\Users\admin\anaconda3\envs\smoke-detect-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1) Load Vintern via huggingface + trust_remote_code
device    = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained("5CD-AI/Vintern-1B-v3_5", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("5CD-AI/Vintern-1B-v3_5", trust_remote_code=True)
model     = AutoModel.from_pretrained(
               "5CD-AI/Vintern-1B-v3_5",
               torch_dtype=torch.bfloat16,
               low_cpu_mem_usage=True,
               trust_remote_code=True,
               use_flash_attn=False,
            ).eval().to(device)

prompt = (
  "<image>\n"
  "Does the image show any person holding a cigarette or joint, or exhaling smoke?"
)

c:\Users\admin\anaconda3\envs\smoke-detect-env\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


FlashAttention2 is not installed.


In [3]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

In [4]:
def build_transform(input_size: int):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

In [5]:
def load_image(image_input, input_size: int = 448):
    """
    Accepts either:
      - a file path (str / Path) → will Image.open(...) it
      - a PIL.Image.Image  → use it directly
    Returns a FloatTensor [1,3,H,W] on CPU.
    """
    if isinstance(image_input, (str, bytes, Path)):
        img = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, Image.Image):
        img = image_input
    else:
        raise ValueError(f"Unsupported type for load_image: {type(image_input)}")

    transform = build_transform(input_size)
    tensor = transform(img).unsqueeze(0)  # add batch
    return tensor

In [6]:
def ask_vintern(frame_bgr):
    # 1) BGR → RGB → PIL
    pil_img = Image.fromarray(
        cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    ).convert("RGB")

    # 2) Build pixel_values using your universal load_image()
    pixel_values = load_image(pil_img).to(device, dtype=torch.bfloat16)

    # 3) Prompt
    prompt = (
        "<image>\n"
        "Does the image show any person holding a cigarette or joint, or exhaling smoke?"
    )

    # 4) Call the exact same chat API from your notebook
    response, _ = model.chat(
        tokenizer,
        pixel_values,
        prompt,
        generation_config={
            "max_new_tokens": 128,
            "do_sample":     False,
            "num_beams":     3,
            "repetition_penalty": 2.5
        },
        history=None,
        return_history=True
    )

    return response.strip().lower()

In [7]:
# 4) Real‐time webcam loop
cap = cv2.VideoCapture(0)
last_check = time.time()
interval   = 3  # seconds
vintern_result = "Waiting…"

cv2.namedWindow("Smoking Detection (Vintern)", cv2.WINDOW_NORMAL)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # only every `interval` seconds do a Vintern query
    if time.time() - last_check >= interval:
        last_check = time.time()
        try:
            out = ask_vintern(frame)
            if any(w in out for w in ["yes"]):
                vintern_result = "Smoking behavior detected"
            else:
                vintern_result = "No smoking behavior detected"
        except Exception as e:
            vintern_result = f"Error: {e}"

    # overlay
    cv2.putText(
        frame, vintern_result, (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2
    )

    cv2.imshow("Smoking Detection (Vintern)", frame)
    if cv2.waitKey(1) & 0xFF in (ord("q"), 27):  # q or ESC
        break
    # also break if user closed the window
    if cv2.getWindowProperty("Smoking Detection (Vintern)", cv2.WND_PROP_VISIBLE) < 1:
        break

cap.release()
cv2.destroyAllWindows()

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
